In [8]:
import pandas as pd

vocab = pd.read_csv("CONCEPT.csv", sep='\t')
vocab.head()

/var/folders/fn/_32pmlmn6zq_hpq142dgnrv00000gq/T/ipykernel_46142/1888869107.py:3: DtypeWarning: Columns (6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  vocab = pd.read_csv("CONCEPT.csv", sep='\t')


,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
0,21600001,ALIMENTARY TRACT AND METABOLISM,Drug,ATC,ATC 1st,C,A,19700101,20991231,NaN
1,21600002,STOMATOLOGICAL PREPARATIONS,Drug,ATC,ATC 2nd,C,A01,19700101,20991231,NaN
2,21600003,STOMATOLOGICAL PREPARATIONS,Drug,ATC,ATC 3rd,C,A01A,19700101,20991231,NaN
3,21600004,Caries prophylactic agents,Drug,ATC,ATC 4th,C,A01AA,19700101,20991231,NaN
4,21600008,"stannous fluoride; oral, local oral",Drug,ATC,ATC 5th,C,A01AA04,19700101,20991231,NaN


In [29]:
# 查询代码511.9对应的concept名称
vocab[vocab['concept_code'] == "E878.8"]['concept_name']

14785    Other specified surgical operations and proced...
Name: concept_name, dtype: object

In [20]:
one_patient = [{'diagnoses': ['4239',
   '5119',
   '78551',
   '4589',
   '7220',
   '2724',
   'E8788',
   'V4501',
   '5119'],
  'procedures': ['3731', '8872', '3491'],
  'medications': ['A02B',
   'B05C',
   'A12A',
   'A12C',
   'C01C',
   'A07A',
   'A10A',
   'N01A',
   'C07A',
   'C03C',
   'A12B',
   'N07A',
   'A03F',
   'A02A',
   'C10A']},
 {'diagnoses': ['4239', '5119', 'V1259', '53081'],
  'procedures': ['8872', '3961'],
  'medications': ['N02B',
   'A02B',
   'A06A',
   'A12C',
   'C01C',
   'A04A',
   'C07A',
   'C03C',
   'A12B',
   'C02D',
   'R01A',
   'C01E',
   'B01A',
   'N05C']}]

In [ ]:
def codes2names(patient_data, vocab):
    """
    Args:
        patient_data: list of dicts, each dict contains 'diagnoses', 'procedures', 'medications'
        vocab: pd.DataFrame, the vocabulary table
    Returns:
        list of dicts, each dict contains 'diagnoses', 'procedures', 'medications'
    """
    converted_patient = []

    for visit in patient_data:
        converted_visit = {}
        
        # convert diagnoses codes
        converted_diagnoses = []
        for code in visit['diagnoses']:
            clean_code = code.replace('.', '')
            # 特殊处理E开头的代码
            if clean_code.startswith('E'):
                if len(clean_code) >= 4:
                    formatted_code = clean_code[:4] + '.' + clean_code[4:]
                else:
                    formatted_code = clean_code
            # 处理普通代码
            else:
                if len(clean_code) >= 3:
                    formatted_code = clean_code[:3] + '.' + clean_code[3:]
                else:
                    formatted_code = clean_code
            concept_match = vocab[vocab['concept_code'] == formatted_code]
            if not concept_match.empty:
                concept_name = concept_match.iloc[0]['concept_name']
                converted_diagnoses.append(concept_name)
            else:
                converted_diagnoses.append(formatted_code)

        # convert procedures codes
        converted_procedures = []
        for code in visit['procedures']:
            clean_code = code.replace('.', '')
            if len(clean_code) >= 2:
                formatted_code = clean_code[:2] + '.' + clean_code[2:]
            else:
                formatted_code = clean_code
            concept_match = vocab[vocab['concept_code'] == formatted_code]
            if not concept_match.empty:
                concept_name = concept_match.iloc[0]['concept_name']
                converted_procedures.append(concept_name)
            else:
                converted_procedures.append(formatted_code)
                
        # convert medications codes
        converted_medications = []
        for code in visit['medications']:
            concept_match = vocab[vocab['concept_code'] == code]
            if not concept_match.empty:
                concept_name = concept_match.iloc[0]['concept_name']
                converted_medications.append(concept_name)
            else:
                converted_medications.append(code)
                
        converted_visit['diagnoses'] = converted_diagnoses
        converted_visit['procedures'] = converted_procedures
        converted_visit['medications'] = converted_medications

        converted_patient.append(converted_visit)

    return converted_patient

In [28]:
converted_patient = codes2names(one_patient, vocab)
converted_patient

[{'diagnoses': ['Unspecified disease of pericardium',
   'Unspecified pleural effusion',
   'Cardiogenic shock',
   'Hypotension, unspecified',
   'Displacement of cervical intervertebral disc without myelopathy',
   'Other and unspecified hyperlipidemia',
   'Other specified surgical operations and procedures causing abnormal patient reaction, or later complication, without mention of misadventure at time of operation',
   'Cardiac pacemaker in situ',
   'Unspecified pleural effusion'],
  'procedures': ['Pericardiectomy',
   'Diagnostic ultrasound of heart',
   'Thoracentesis'],
  'medications': ['DRUGS FOR PEPTIC ULCER AND GASTRO-OESOPHAGEAL REFLUX DISEASE (GORD)',
   'IRRIGATING SOLUTIONS',
   'CALCIUM',
   'OTHER MINERAL SUPPLEMENTS',
   'CARDIAC STIMULANTS EXCL. CARDIAC GLYCOSIDES',
   'INTESTINAL ANTIINFECTIVES',
   'INSULINS AND ANALOGUES',
   'ANESTHETICS, GENERAL',
   'BETA BLOCKING AGENTS',
   'HIGH-CEILING DIURETICS',
   'POTASSIUM',
   'PARASYMPATHOMIMETICS',
   'PROPULSIVE